Import all necessary libraries.

In [1]:
import pandas as pd
import numpy as np
import os
import wandb
import random
import math
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader
from imblearn.metrics import geometric_mean_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from types import SimpleNamespace

from sklearn.metrics import confusion_matrix
import seaborn as sns

import sys
sys.path.append(os.path.join(os.getcwd(), '../src'))

from transforms.feature_engineering_classification import add_all_features, filter_business_hours, entries_per_day_per_site
from transforms.feature_engineering_classification import (
    CONTINUOUS_FEATURE_COLUMNS,
    CATEGORICAL_FEATURE_COLUMNS,
    CYCLIC_FEATURE_COLUMNS,
    TARGET_COLUMN
)
from evaluation.comp_metrics import evaluate_all_metrics
from evaluation.visual import plot_confusion_matrix

from datasets.flextrack_dataset import FlextrackClassificationDataset
from utils.losses import FocalLoss

SWEEP = True

c:\Users\timon\.pyenv-win-venv\envs\aicomp\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\timon\.pyenv-win-venv\envs\aicomp\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because a

Set seed for reproducibility.

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
os.environ["WANDB_API_KEY"] = "3aaf9f796df65417b3f5f8560b43875171b55805"

wandb.login()

wandb: Currently logged in as: fabian-dubach (fabian-dubach-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
# just use regression data and remove DR-Flags and DR-Capacity
df_train = pd.read_csv(os.path.normpath(os.path.join(os.getcwd(), '../data/regression/regression-train.csv')))
df_test = pd.read_csv(os.path.normpath(os.path.join(os.getcwd(), '../data/regression/regression-test.csv')))

In [5]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

Dataset shape: (105120, 7)

Columns: ['Site', 'Timestamp_Local', 'Dry_Bulb_Temperature_C', 'Global_Horizontal_Radiation_W/m2', 'Building_Power_kW', 'Demand_Response_Flag', 'Demand_Response_Capacity_kW']


# Feature Engineering

We use same feature engineering as in regression task but remove the irrelevant features.

In [6]:
df_train = add_all_features(df_train)

df_train = filter_business_hours(df_train)

ENTRIES_PER_DAY = entries_per_day_per_site(df_train)

In [7]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

Dataset shape: (58035, 40)

Columns: ['Site', 'Timestamp_Local', 'Dry_Bulb_Temperature_C', 'Global_Horizontal_Radiation_W/m2', 'Building_Power_kW', 'Demand_Response_Flag', 'Demand_Response_Capacity_kW', 'hour', 'minute', 'day_of_week', 'is_weekend', 'is_holiday', 'month_sin', 'month_cos', 'Building_Power_kW_diff_15min', 'Building_Power_kW_diff_1h', 'Building_Power_kW_diff_1d', 'Dry_Bulb_Temperature_C_diff_15min', 'Global_Horizontal_Radiation_W/m2_diff_15min', 'Building_Power_kW_rolling_mean_1h', 'Building_Power_kW_rolling_mean_2h', 'Building_Power_kW_rolling_mean_1d', 'Building_Power_kW_rolling_std_1h', 'Building_Power_kW_rolling_std_2h', 'Building_Power_kW_rolling_std_1d', 'Building_Power_kW_rolling_min_1h', 'Building_Power_kW_rolling_min_2h', 'Building_Power_kW_rolling_max_1h', 'Building_Power_kW_rolling_max_2h', 'minute_0', 'minute_15', 'minute_30', 'minute_45', 'day_of_week_0', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'day_of_week_5', 'day_of_week_6']


# Split sites

In [8]:
def count_sites(df):

    counter_site_a = 0
    counter_site_b = 0
    counter_site_c = 0
    counter_site_d = 0
    counter_site_e = 0

    for i in df['Site']:
        if i == 'siteA':
            counter_site_a += 1
        elif i == 'siteB':
            counter_site_b += 1
        elif i == 'siteC':
            counter_site_c += 1
        elif i == 'siteD':
            counter_site_d += 1
        elif i == 'siteE':
            counter_site_e += 1
    
    return counter_site_a, counter_site_b, counter_site_c, counter_site_d, counter_site_e

In [9]:
df_train_site_a = df_train[0:19345]
count_sites(df_train_site_a)

df_train_site_b = df_train[19345:38690]
count_sites(df_train_site_b)

df_train_site_c = df_train[38690:58035]
count_sites(df_train_site_c)

(0, 0, 19345, 0, 0)

In [10]:
X_continuous_site_a = df_train_site_a[CONTINUOUS_FEATURE_COLUMNS].values # Convert to numpy array
X_continuous_site_b = df_train_site_b[CONTINUOUS_FEATURE_COLUMNS].values # Convert to numpy array
X_continuous_site_c = df_train_site_c[CONTINUOUS_FEATURE_COLUMNS].values # Convert to numpy array

X_categorical_site_a = df_train_site_a[CATEGORICAL_FEATURE_COLUMNS].values # Convert to numpy array
X_categorical_site_b = df_train_site_b[CATEGORICAL_FEATURE_COLUMNS].values # Convert to numpy array
X_categorical_site_c = df_train_site_c[CATEGORICAL_FEATURE_COLUMNS].values # Convert to numpy array

X_cyclic_site_a = df_train_site_a[CYCLIC_FEATURE_COLUMNS].values # Convert to numpy array
X_cyclic_site_b = df_train_site_b[CYCLIC_FEATURE_COLUMNS].values # Convert to numpy array
X_cyclic_site_c = df_train_site_c[CYCLIC_FEATURE_COLUMNS].values # Convert to numpy array

y_site_a = df_train_site_a[TARGET_COLUMN].values.reshape(-1, 1) # Convert to numpy array and reshape
y_site_b = df_train_site_b[TARGET_COLUMN].values.reshape(-1, 1) # Convert to numpy array and reshape
y_site_c = df_train_site_c[TARGET_COLUMN].values.reshape(-1, 1) # Convert to numpy array and reshape

In [11]:
print(f"Continuous feature shape: {X_continuous_site_a.shape}")
print(f"Continuous feature shape: {X_continuous_site_b.shape}")
print(f"Continuous feature shape: {X_continuous_site_c.shape}")

print(f"Categorical feature shape: {X_categorical_site_a.shape}")
print(f"Categorical feature shape: {X_categorical_site_b.shape}")
print(f"Categorical feature shape: {X_categorical_site_c.shape}")

print(f"Cyclic feature shape: {X_cyclic_site_a.shape}")
print(f"Cyclic feature shape: {X_cyclic_site_b.shape}")
print(f"Cyclic feature shape: {X_cyclic_site_c.shape}")

print(f"Target shape: {y_site_a.shape}")
print(f"Target shape: {y_site_b.shape}")
print(f"Target shape: {y_site_c.shape}")

Continuous feature shape: (19345, 19)
Continuous feature shape: (19345, 19)
Continuous feature shape: (19345, 19)
Categorical feature shape: (19345, 13)
Categorical feature shape: (19345, 13)
Categorical feature shape: (19345, 13)
Cyclic feature shape: (19345, 2)
Cyclic feature shape: (19345, 2)
Cyclic feature shape: (19345, 2)
Target shape: (19345, 1)
Target shape: (19345, 1)
Target shape: (19345, 1)


In [12]:
y_site_a_unscaled = y_site_a.copy()

# Normalization

In [13]:
scaler_X_site_a = StandardScaler()
scaler_X_site_b = StandardScaler()
scaler_X_site_c = StandardScaler()

In [14]:
X_scaled_site_a = scaler_X_site_a.fit_transform(X_continuous_site_a)
X_scaled_site_b = scaler_X_site_b.fit_transform(X_continuous_site_b)
X_scaled_site_c = scaler_X_site_c.fit_transform(X_continuous_site_c)

Concatenate the unscaled and the scaled features together.

In [15]:
X_site_a = np.concatenate([X_scaled_site_a, X_categorical_site_a, X_cyclic_site_a], axis=1)
X_site_b = np.concatenate([X_scaled_site_b, X_categorical_site_b, X_cyclic_site_b], axis=1)
X_site_c = np.concatenate([X_scaled_site_c, X_categorical_site_c, X_cyclic_site_c], axis=1)

In [16]:
X_site_a = X_site_a.astype(np.float32)
X_site_b = X_site_b.astype(np.float32)
X_site_c = X_site_c.astype(np.float32)

y_site_a = y_site_a.astype(int)
y_site_b = y_site_b.astype(int)
y_site_c = y_site_c.astype(int)

In [17]:
print(f"Continuous feature shape: {X_site_a.shape}")
print("First few entries of each site have nan values due to feature engineering:\n", X_site_a[0])
print(X_site_a[ENTRIES_PER_DAY])

Continuous feature shape: (19345, 34)
First few entries of each site have nan values due to feature engineering:
 [ 0.57910997 -1.3638599  -0.5221737  -0.00325376 -0.00751908         nan
 -0.69390106  0.0624692  -0.54411376 -0.5452835          nan -0.64983785
 -0.84162885         nan -0.38954532 -0.25569385 -0.6574576  -0.75812143
 -1.602483    1.          0.          0.          0.          0.
  1.          0.          0.          0.          0.          0.
  0.          1.          0.5         0.8660254 ]
[ 3.1494236e-01 -1.3638599e+00 -5.2217370e-01 -3.2537556e-03
 -7.5190784e-03  9.0518305e-03 -3.7747535e-01  6.2469199e-02
 -5.4411376e-01 -5.4528350e-01  2.9363585e+00 -6.4983785e-01
 -8.4162885e-01  4.3428288e+00 -3.8954532e-01 -2.5569385e-01
 -6.5745759e-01 -7.5812143e-01 -1.6024830e+00  1.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  1.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00

In [18]:
np.unique(y_site_a)

array([0, 1, 2])

IMPORTANT: Remove entries, where features are incomplete (at start of dataset)

In [19]:
# Create mask to exclude first ENTRIES_PER_DAY of each site
mask_site_a = np.ones(len(X_site_a), dtype=bool)
mask_site_b = np.ones(len(X_site_b), dtype=bool)
mask_site_c = np.ones(len(X_site_c), dtype=bool)

# Site A: exclude indices 0 to ENTRIES_PER_DAY-1
mask_site_a[0:ENTRIES_PER_DAY] = False

# Site B: exclude indices (365*ENTRIES_PER_DAY) to (365*ENTRIES_PER_DAY + ENTRIES_PER_DAY-1)
mask_site_b[0:ENTRIES_PER_DAY] = False

# Site C: exclude indices (730*ENTRIES_PER_DAY) to (730*ENTRIES_PER_DAY + ENTRIES_PER_DAY-1)
mask_site_c[0:ENTRIES_PER_DAY] = False

# Apply mask to remove incomplete entries
X_site_a = X_site_a[mask_site_a]
X_site_b = X_site_b[mask_site_b]
X_site_c = X_site_c[mask_site_c]

y_site_a = y_site_a[mask_site_a]
y_site_b = y_site_b[mask_site_b]
y_site_c = y_site_c[mask_site_c]

### Data Splitting

In [20]:
# Calculate split indices (accounting for removed incomplete entries)
site_a_entries = (365 - 1) * ENTRIES_PER_DAY  # 364 days of site A
site_b_entries = (365 - 1) * ENTRIES_PER_DAY  # 364 days of site B

# Train on Site A and Site C, validate on Site B
X_train = np.vstack((X_site_a, X_site_c))
X_val = X_site_b
y_train = np.vstack((y_site_a, y_site_c))
y_val = y_site_b

In [21]:
print(len(X_train))
print(len(y_train))
print(len(X_val))
print(len(y_val))

38584
38584
19292
19292


# Parameters

In [22]:
config = {
    # Model hyperparameters
    'input_size': X_train.shape[1],
    'embedding_dim': 64,
    'num_layers': 2,
    'nhead': 4,
    'dropout': 0.3,
    'sequence_length': 53, # Full Day
    
    # Training hyperparameters
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'batch_size': 32,
    'num_epochs': 50,
    'gradient_clip_val': 1.0,
    'warmup_epochs': 5,
    'optimizer': 'Adam',

    # Loss Function Parameters
    'loss_function': 'FocalLoss', # CrossEntropy, WeightedCrossEntropy, FocalLoss
    'focal_alpha': 1,
    'focal_gamma': 2,
    
    # Model architecture
    'model_type': 'Transformer' # TFT (Temporal Fusion Transformer)
}

config = SimpleNamespace(**config)

## Prepare Sequences

Create sequences to create a "sliding window" for the RNN architechture to predict the current hidden state based on the past values.

In [23]:
def create_sequences(X, y, seq_length=config.sequence_length):
    sequences_X = []
    sequences_y = []
    
    for i in range(len(X) - seq_length):
        sequences_X.append(X[i:i+seq_length])
        sequences_y.append(y[i+seq_length])
    
    return np.array(sequences_X), np.array(sequences_y)

In [24]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), 0, :].unsqueeze(0)
        return self.dropout(x)


class TransformerClassifier(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, 
                 num_classes=3, dropout=0.3):
        super().__init__()
        
        assert d_model % nhead == 0, f"d_model ({d_model}) must be divisible by nhead ({nhead})"
        
        self.embedding = nn.Linear(input_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, 
            nhead, 
            dim_feedforward=d_model*4,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.transformer(x)
        x = x[:, -1, :]
        x = self.norm(x)
        x = self.dropout(x)
        return self.classifier(x)

Define hyperparameters for the model.

Define cuda as the device to make the training possible to the GPU.

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# Training

In [26]:
# Warmup scheduler function
def get_lr_with_warmup(epoch, base_lr, warmup_epochs):
    """
    Calculate learning rate with linear warmup
    
    Args:
        epoch: Current epoch (0-indexed)
        base_lr: Target learning rate after warmup
        warmup_epochs: Number of epochs for warmup
    
    Returns:
        Current learning rate
    """
    if warmup_epochs == 0 or epoch >= warmup_epochs:
        return base_lr
    else:
        # Linear warmup from 0 to base_lr
        return base_lr * (epoch + 1) / warmup_epochs

In [27]:
# CrossEntropy, WeightedCrossEntropy, FocalLoss
def get_loss_function(config, y_train_seq):
    
    if config.loss_function == 'WeightedCrossEntropy':
        class_counts = np.bincount(y_train_seq.flatten())
        class_weights = 1.0 / class_counts
        class_weights = torch.FloatTensor(class_weights).to(device)
        return nn.CrossEntropyLoss(weight=class_weights)

    elif config.loss_function == 'FocalLoss':
        return FocalLoss(alpha=config.focal_alpha, gamma=config.focal_gamma)
    
    else:
        return nn.CrossEntropyLoss()

In [28]:
def train(config=None):

    if config is not None:
        wandb.init(
            # Set the wandb entity where your project will be logged (generally your team name).
            entity="fabian-dubach-hochschule-luzern",
            # Set the wandb project where this run will be logged.
            project="AICOMP_Flextrack",
            # Name this run
            name=f"{config.model_type}-{config.loss_function}-classification-emb_{config.embedding_dim}-h_{config.nhead}-l_{config.num_layers}-seq_{config.sequence_length}-lr_{config.learning_rate:.0e}",
            # Track hyperparameters and run metadata.
            config=config
        )
    else:
        wandb.init(
            project="AICOMP_Flextrack",
            entity="fabian-dubach-hochschule-luzern"
        )
        config = wandb.config
        wandb.run.name = f"{config.model_type}-{config.loss_function}-classification-emb_{config.embedding_dim}-h_{config.nhead}-l_{config.num_layers}-seq_{config.sequence_length}-lr_{config.learning_rate:.0e}"

    print("WandB initialized successfully!")

    # ============== RECREATE SEQUENCES ==============
    # Important: Sequences depend on sequence_length!
    X_train_seq_a, y_train_seq_a = create_sequences(X_site_a, y_site_a, config.sequence_length)
    X_train_seq_c, y_train_seq_c = create_sequences(X_site_c, y_site_c, config.sequence_length)
    X_val_seq, y_val_seq = create_sequences(X_val, y_val, config.sequence_length)

    X_train_seq = np.vstack((X_train_seq_a, X_train_seq_c))
    y_train_seq = np.vstack((y_train_seq_a, y_train_seq_c))

    train_dataset = FlextrackClassificationDataset(X_train_seq, y_train_seq)
    val_dataset = FlextrackClassificationDataset(X_val_seq, y_val_seq)

    batch_size = config.batch_size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # ============== CREATE MODEL ==============
    model = TransformerClassifier(
        input_size=config.input_size,
        d_model=config.embedding_dim,
        nhead=config.nhead,
        num_layers=config.num_layers,
        dropout=config.dropout,  # Now included!
        num_classes=3
    ).to(device)
    print(f"Model architecture:\n{model}")

    # weighted loss
    criterion = get_loss_function(config, y_train_seq)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')

    print("Starting training...")
    print(f"Warmup enabled: {config.warmup_epochs} epochs")

    for epoch in range(config.num_epochs):

        # ============== LEARNING RATE WARMUP ==============
        current_lr = get_lr_with_warmup(
            epoch,
            config.learning_rate,
            config.warmup_epochs
        )

        for param_group in optimizer.param_groups:
            param_group['lr'] = current_lr

        # ============== TRAINING ==============
        model.train()
        train_loss = 0
        train_preds = []
        train_targets = []

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # Forward
            logits = model(X_batch)  # shape: (batch, 3)
            loss = criterion(logits, y_batch)  # CE expects integer labels

            # Backward
            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(model.parameters(), config.gradient_clip_val)
            optimizer.step()

            train_loss += loss.item()

            # Store predictions + targets
            preds = torch.argmax(logits, dim=1)
            train_preds.append(preds.cpu().numpy())
            train_targets.append(y_batch.cpu().numpy())

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        train_preds = np.concatenate(train_preds)
        train_targets = np.concatenate(train_targets)

        # Classification metrics
        train_gmean = geometric_mean_score(train_targets, train_preds)
        train_f1 = f1_score(train_targets, train_preds, average="macro")

        # ============== VALIDATION ==============
        model.eval()
        val_loss = 0
        val_preds = []
        val_targets = []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_loss += loss.item()

                preds = torch.argmax(logits, dim=1)
                val_preds.append(preds.cpu().numpy())
                val_targets.append(y_batch.cpu().numpy())

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        val_preds = np.concatenate(val_preds)
        val_targets = np.concatenate(val_targets)

        val_gmean = geometric_mean_score(val_targets, val_preds)
        val_f1 = f1_score(val_targets, val_preds, average="macro")

        # Create confusion matrices
        train_cm_fig = plot_confusion_matrix(train_targets, train_preds, 
                                            title=f'Training Confusion Matrix - Epoch {epoch+1}')
        val_cm_fig = plot_confusion_matrix(val_targets, val_preds,
                                            title=f'Validation Confusion Matrix - Epoch {epoch+1}')
        # wandb logging
        wandb.log({
            "epoch": epoch,
            "lr": current_lr,
            "train/loss": train_loss,
            "val/loss": val_loss,

            # training metrics
            "train/geometric_mean": train_gmean,
            "train/f1": train_f1,
            "train/confusion_matrix": wandb.Image(train_cm_fig),

            # validation metrics
            "val/geometric_mean": val_gmean,
            "val/f1": val_f1,
            "val/confusion_matrix": wandb.Image(val_cm_fig),
        })
        
        # Close figures to free memory
        plt.close(train_cm_fig)
        plt.close(val_cm_fig)

        # ============== PRINT PROGRESS ==============
        if (epoch + 1) % 10 == 0:
            print(f"\nEpoch [{epoch+1}/{config.num_epochs}]")
            print(f"LR: {current_lr:.6f}")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"Train   - GMean: {train_gmean:.4f} | F1: {train_f1:.4f}")
            print(f"Val     - GMean: {val_gmean:.4f} | F1: {val_f1:.4f}")

        # save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pt")

    wandb.finish()
    print("\nTraining completed!")

# Sweep

In [ ]:
sweep_config = {
    'method': 'bayes',  # Bayesian optimization is more efficient than grid/random
    'metric': {
        'name': 'val/f1',
        'goal': 'maximize'
    },
    'parameters': {
        # ============== STATIC (Not Tuned) ==============
        'input_size': {'value': X_train.shape[1]},
        'batch_size': {'value': 32},
        'num_epochs': {'value': 100},
        'optimizer': {'value': 'Adam'},
        'model_type': {'value': 'Transformer'},
        
        # ============== ARCHITECTURE ==============
        'embedding_dim': {
            'values': [64, 128, 256]  # Model capacity
        },
        'num_layers': {
            'values': [2, 3, 4]  # Depth
        },
        'nhead': {
            'values': [4, 8]  # Must divide embedding_dim evenly
            # 4 heads: 64/4=16, 128/4=32, 256/4=64
            # 8 heads: 64/8=8, 128/8=16, 256/8=32
        },
        'dropout': {
            'distribution': 'uniform',
            'min': 0.1,
            'max': 0.5
        },
        'sequence_length': {
            'values': [24, 36, 53]  # 6h, 9h, full day
        },
        
        # ============== OPTIMIZATION ==============
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 1e-5,
            'max': 1e-3
        },
        'weight_decay': {
            'distribution': 'log_uniform_values',
            'min': 1e-6,
            'max': 1e-4
        },
        'gradient_clip_val': {
            'values': [0.5, 1.0, 2.0]
        },
        'warmup_epochs': {
            'values': [0, 3, 5, 10]  # 0 = no warmup
        },
        
        # ============== LOSS FUNCTION ==============
        'loss_function': {
            'values': ['CrossEntropy', 'WeightedCrossEntropy', 'FocalLoss']
        },
        'focal_alpha': {
            'distribution': 'uniform',
            'min': 0.25,
            'max': 2.0
        },
        'focal_gamma': {
            'distribution': 'uniform',
            'min': 1.0,
            'max': 5.0
        }
    },
}

: 

In [ ]:
if SWEEP:
    sweep_id = wandb.sweep(
        sweep_config, 
        project="AICOMP_Flextrack",
        entity="fabian-dubach-hochschule-luzern"
    )

    wandb.agent(sweep_id, function=train, count=30)  # Try 30 combinations
else:
    train(config)

Create sweep with ID: 5kyuaoea
Sweep URL: https://wandb.ai/fabian-dubach-hochschule-luzern/AICOMP_Flextrack/sweeps/5kyuaoea


wandb: Agent Starting Run: ur84bpyn with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.29694505523560055
wandb: 	embedding_dim: 128
wandb: 	focal_alpha: 0.7622485691206622
wandb: 	focal_gamma: 3.677138325515358
wandb: 	gradient_clip_val: 1
wandb: 	input_size: 34
wandb: 	learning_rate: 0.0007731021215924142
wandb: 	loss_function: FocalLoss
wandb: 	model_type: Transformer
wandb: 	nhead: 8
wandb: 	num_epochs: 100
wandb: 	num_layers: 3
wandb: 	optimizer: Adam
wandb: 	sequence_length: 24
wandb: 	warmup_epochs: 3
wandb: 	weight_decay: 8.65662827912283e-05


WandB initialized successfully!
Model architecture:
TransformerClassifier(
  (embedding): Linear(in_features=34, out_features=128, bias=True)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.29694505523560055, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.29694505523560055, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.29694505523560055, inplace=False)
        (dropout2): Dropout(p=0.29694505523560055, inplace=False)
      )
    )
  

wandb: ERROR The nbformat package was not found. It is required to save notebook history.



Epoch [100/100]
LR: 0.000773
Train Loss: 0.0233 | Val Loss: 0.0475
Train   - GMean: 0.0479 | F1: 0.3381
Val     - GMean: 0.0000 | F1: 0.4282


...


Epoch [100/100]
LR: 0.000012
Train Loss: 0.0376 | Val Loss: 0.3570
Train   - GMean: 0.6417 | F1: 0.7759
Val     - GMean: 0.5776 | F1: 0.5596


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇████
lr,▁███████████████████████████████████████
train/f1,▁▁▁▁▁▂▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█████████
train/geometric_mean,▁▁▁▁▁▂▂▂▂▃▃▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████
train/loss,█▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val/f1,▁▁▁▁▁▁▃▅▅▅▅▅▆▆▇█████▇████▇▇█████████████
val/geometric_mean,▁▁▁▁▁▃▃▃▄▄▅▅▇▇▇▇▇▇▇▇▇▇██████████████████
val/loss,▃▂▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▆▅▅▅▅▆▆▆▆▇▇▇▇██▇▇█▇▇
epoch,99
lr,1e-05
train/f1,0.7759



Training completed!


wandb: Agent Starting Run: igr710e8 with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.11014230072990344
wandb: 	embedding_dim: 128
wandb: 	focal_alpha: 1.3022968979525176
wandb: 	focal_gamma: 4.4667333351767144
wandb: 	gradient_clip_val: 1
wandb: 	input_size: 34
wandb: 	learning_rate: 3.308088492503773e-05
wandb: 	loss_function: WeightedCrossEntropy
wandb: 	model_type: Transformer
wandb: 	nhead: 4
wandb: 	num_epochs: 100
wandb: 	num_layers: 4
wandb: 	optimizer: Adam
wandb: 	sequence_length: 53
wandb: 	warmup_epochs: 10
wandb: 	weight_decay: 3.115823339722463e-06


WandB initialized successfully!
Model architecture:
TransformerClassifier(
  (embedding): Linear(in_features=34, out_features=128, bias=True)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.11014230072990344, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.11014230072990344, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.11014230072990344, inplace=False)
        (dropout2): Dropout(p=0.11014230072990344, inplace=False)
      )
    )
  